<a href="https://colab.research.google.com/github/kalsep/Data-Engineering-Interview-Preperations/blob/main/Pyspark_Practical_Interview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col,unix_timestamp,avg,when

In [ ]:
spark = SparkSession.builder.appName("PySpark Practice").config("spark.some.config.option", "some-value").getOrCreate()

# Question 1
Given employer history data, where each record contains details about an employee’s work history — employer, job position, and the start and end dates of each job.

We need to find out how many users had Microsoft as their employer, and immediately after that, they started working at Google, with no other employers between these two positions.

In [ ]:
linkedin_data = [
    (1, 'Microsoft', 'developer', '2020-04-13', '2021-11-01'),
    (1, 'Google', 'developer', '2021-11-01', None),
    (2, 'Google', 'manager', '2021-01-01', '2021-01-11'),
    (2, 'Microsoft', 'manager', '2021-01-11', None),
    (3, 'Microsoft', 'analyst', '2019-03-15', '2020-07-24'),
    (3, 'Amazon', 'analyst', '2020-08-01', '2020-11-01'),
    (3, 'Google', 'senior analyst', '2020-11-01', '2021-03-04'),
    (4, 'Google', 'junior developer', '2018-06-01', '2021-11-01'),
    (4, 'Google', 'senior developer', '2021-11-01', None),
    (5, 'Microsoft', 'manager', '2017-09-26', None),
    (6, 'Google', 'CEO', '2015-10-02', None)
]

linkedin_columns = [
  'emp_id',
  'employer',
  'position',
  'start_date',
  'end_date'
]
# Create a DataFrame
linkedin_df = spark.createDataFrame(linkedin_data, linkedin_columns)

In [ ]:
linkedin_df.show()

+------+---------+----------------+----------+----------+
|emp_id| employer|        position|start_date|  end_date|
+------+---------+----------------+----------+----------+
|     1|Microsoft|       developer|2020-04-13|2021-11-01|
|     1|   Google|       developer|2021-11-01|      NULL|
|     2|   Google|         manager|2021-01-01|2021-01-11|
|     2|Microsoft|         manager|2021-01-11|      NULL|
|     3|Microsoft|         analyst|2019-03-15|2020-07-24|
|     3|   Amazon|         analyst|2020-08-01|2020-11-01|
|     3|   Google|  senior analyst|2020-11-01|2021-03-04|
|     4|   Google|junior developer|2018-06-01|2021-11-01|
|     4|   Google|senior developer|2021-11-01|      NULL|
|     5|Microsoft|         manager|2017-09-26|      NULL|
|     6|   Google|             CEO|2015-10-02|      NULL|
+------+---------+----------------+----------+----------+



In [ ]:
window_spec = Window.partitionBy('emp_id').orderBy('start_date')

In [ ]:
# Use LEAD function to get the next employer for each user
linkedin_with_next_employer = linkedin_df.withColumn('next_employer', F.lead('employer').over(window_spec))

In [ ]:
linkedin_with_next_employer.show()

+------+---------+----------------+----------+----------+-------------+
|emp_id| employer|        position|start_date|  end_date|next_employer|
+------+---------+----------------+----------+----------+-------------+
|     1|Microsoft|       developer|2020-04-13|2021-11-01|       Google|
|     1|   Google|       developer|2021-11-01|      NULL|         NULL|
|     2|   Google|         manager|2021-01-01|2021-01-11|    Microsoft|
|     2|Microsoft|         manager|2021-01-11|      NULL|         NULL|
|     3|Microsoft|         analyst|2019-03-15|2020-07-24|       Amazon|
|     3|   Amazon|         analyst|2020-08-01|2020-11-01|       Google|
|     3|   Google|  senior analyst|2020-11-01|2021-03-04|         NULL|
|     4|   Google|junior developer|2018-06-01|2021-11-01|       Google|
|     4|   Google|senior developer|2021-11-01|      NULL|         NULL|
|     5|Microsoft|         manager|2017-09-26|      NULL|         NULL|
|     6|   Google|             CEO|2015-10-02|      NULL|       

In [ ]:
# Filter to find users who worked at Microsoft and then moved to Google
result = linkedin_with_next_employer.filter(
    (linkedin_with_next_employer.employer == 'Microsoft') &
    (linkedin_with_next_employer.next_employer == 'Google')
)
# Select emp_id(s)
emp_ids = result.select('emp_id').distinct()

# Show the result
emp_ids.show()

# Count the number of distinct user_ids
user_count = emp_ids.count()
print(f"Number of users who worked at Microsoft and then moved to Google: {user_count}")

+------+
|emp_id|
+------+
|     1|
+------+

Number of users who worked at Microsoft and then moved to Google: 1


# Question 2
Given a table of hotels with various attributes (hotel_address, additional_number_of_scoring, review_date, average_score, hotel_name, reviewer_nationality, negative_review, review_total_negative_word_counts, total_number_of_reviews, positive_review, review_total_positive_word_counts, total_number_of_reviews_reviewer_has_given, reviewer_score, tags, days_since_review, lat, lng ), We need to find the top 10 hotels with the highest average scores. The output should include:

- The hotel name.
- The average score of the hotel.
- The records should be sorted by average score in descending order.
### Solution
We can solve this problem step-by-step using PySpark:

Read the Hotel Data: We’ll start by creating a DataFrame from the provided hotel data.
Calculate the Average Score: We will use the avg() function to calculate the average score of each hotel. Since the average_score column already contains the average score, we can directly use it.
Sort the Hotels by Score: After calculating the average scores, we will order the hotels in descending order of their average score.
Limit to Top 10: Finally, we will use limit(10) to get the top 10 hotels based on the highest ratings.
PySpark Code Implementation

In [ ]:
data = [
  ('123 Ocean Ave, Miami, FL', 3, '2024-11-10', 4.2, 'Ocean View', 'American', 'Room small, but clean.', 5, 150, 'Great location and friendly staff!', 8, 30, 4.5, 'beachfront, family-friendly', '5 days', 25.7617, -80.1918),
  ('456 Mountain Rd, Boulder, CO', 2, '2024-11-12', 3.9, 'Mountain Lodge', 'Canadian', 'wifi slow.', 3, 120, 'nice rooms.', 10, 20, 4.0, 'scenic, nature', '3 days', 40.015, -105.2705),
  ('789 Downtown St, New York, NY', 5, '2024-11-15', 4.7, 'Central Park Hotel', 'British', 'Noisy, sleep.', 7, 200, 'Perfect location near Central Park.', 12, 50, 4.7, 'luxury, city-center', '1 day', 40.7831, -73.9712),
  ('101 Lakeside Blvd, Austin, TX', 1, '2024-11-08', 4.0, 'Lakeside Inn', 'Mexican', 'food avg.', 4, 80, 'Nice, friendly service.', 6, 15, 3.8, 'relaxing, family', '10 days', 30.2672, -97.7431),
  ('202 River Ave, Nashville, TN', 4, '2024-11-13', 4.5, 'Riverside', 'German', 'Limited parking', 2, 175, 'Great rooms.', 9, 25, 4.2, 'riverfront, peaceful', '2 days', 36.1627, -86.7816)
]
# Define columns for the hotel DataFrame
columns = [
  "hotel_address",
  "additional_number_of_scoring",
  "review_date",
  "average_score",
  "hotel_name",
  "reviewer_nationality",
  "negative_review",
  "review_total_negative_word_counts",
  "total_number_of_reviews",
  "positive_review",
  "review_total_positive_word_counts",
  "total_number_of_reviews_reviewer_has_given",
  "reviewer_score",
  "tags",
  "days_since_review",
  "lat",
  "lng"
]

In [ ]:
hotel_df = spark.createDataFrame(data, columns)

In [ ]:
hotel_df.show()

+--------------------+----------------------------+-----------+-------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------------------+--------------+--------------------+-----------------+-------+---------+
|       hotel_address|additional_number_of_scoring|review_date|average_score|        hotel_name|reviewer_nationality|     negative_review|review_total_negative_word_counts|total_number_of_reviews|     positive_review|review_total_positive_word_counts|total_number_of_reviews_reviewer_has_given|reviewer_score|                tags|days_since_review|    lat|      lng|
+--------------------+----------------------------+-----------+-------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+---------------

In [ ]:
hotel_df.select('hotel_name','average_score').orderBy(col("average_score").desc()).show(n=10)

+------------------+-------------+
|        hotel_name|average_score|
+------------------+-------------+
|Central Park Hotel|          4.7|
|         Riverside|          4.5|
|        Ocean View|          4.2|
|      Lakeside Inn|          4.0|
|    Mountain Lodge|          3.9|
+------------------+-------------+



# Question 3
We have a table of employees that includes the following fields: id, first_name, last_name, age, sex, employee_title, department, salary, target, bonus, city, address, and manager_id. We need to find the top 3 distinct salaries for each department. The output should include:

- The department name.
- The top 3 distinct salaries for each department.
- The results should be ordered alphabetically by department and then by the highest salary to the lowest salary.

In [ ]:
data = [
    (1, 'Allen', 'Wang', 55, 'F', 'Manager', 'Management', 200000, 0, 300, 'California', '23St', 1),
    (13, 'Katty', 'Bond', 56, 'F', 'Manager', 'Management', 150000, 0, 300, 'Arizona', None, 1),
    (19, 'George', 'Joe', 50, 'M', 'Manager', 'Management', 100000, 0, 300, 'Florida', '26St', 1),
    (11, 'Richerd', 'Gear', 57, 'M', 'Manager', 'Management', 250000, 0, 300, 'Alabama', None, 1),
    (10, 'Jennifer', 'Dion', 34, 'F', 'Sales', 'Sales', 100000, 200, 150, 'Alabama', None, 13),
    (18, 'Laila', 'Mark', 26, 'F', 'Sales', 'Sales', 100000, 200, 150, 'Florida', '23St', 11),
    (20, 'Sarrah', 'Bicky', 31, 'F', 'Senior Sales', 'Sales', 200000, 200, 150, 'Florida', '53St', 19),
    (21, 'Suzan', 'Lee', 34, 'F', 'Sales', 'Sales', 130000, 200, 150, 'Florida', '56St', 19),
    (22, 'Mandy', 'John', 31, 'F', 'Sales', 'Sales', 130000, 200, 150, 'Florida', '45St', 19),
    (17, 'Mick', 'Berry', 44, 'M', 'Senior Sales', 'Sales', 220000, 200, 150, 'Florida', None, 11),
    (12, 'Shandler', 'Bing', 23, 'M', 'Auditor', 'Audit', 110000, 200, 150, 'Arizona', None, 11),
    (14, 'Jason', 'Tom', 23, 'M', 'Auditor', 'Audit', 100000, 200, 150, 'Arizona', None, 11),
    (16, 'Celine', 'Anston', 27, 'F', 'Auditor', 'Audit', 100000, 200, 150, 'Colorado', None, 11),
    (15, 'Michale', 'Jackson', 44, 'F', 'Auditor', 'Audit', 70000, 150, 150, 'Colorado', None, 11),
    (6, 'Molly', 'Sam', 28, 'F', 'Sales', 'Sales', 140000, 100, 150, 'Arizona', '24St', 13),
    (7, 'Nicky', 'Bat', 33, 'F', 'Sales', 'Sales', None, None, None, None, None, None)
]
# Define columns for the employees DataFrame
columns = [
  "id",
  "first_name",
  "last_name",
  "age",
  "sex",
  "employee_title",
  "department",
  "salary",
  "target",
  "bonus", "city",
  "address",
  "manager_id"
]
# Create DataFrame
df = spark.createDataFrame(data, columns)


In [ ]:
df.show()

+---+----------+---------+---+---+--------------+----------+------+------+-----+----------+-------+----------+
| id|first_name|last_name|age|sex|employee_title|department|salary|target|bonus|      city|address|manager_id|
+---+----------+---------+---+---+--------------+----------+------+------+-----+----------+-------+----------+
|  1|     Allen|     Wang| 55|  F|       Manager|Management|200000|     0|  300|California|   23St|         1|
| 13|     Katty|     Bond| 56|  F|       Manager|Management|150000|     0|  300|   Arizona|   NULL|         1|
| 19|    George|      Joe| 50|  M|       Manager|Management|100000|     0|  300|   Florida|   26St|         1|
| 11|   Richerd|     Gear| 57|  M|       Manager|Management|250000|     0|  300|   Alabama|   NULL|         1|
| 10|  Jennifer|     Dion| 34|  F|         Sales|     Sales|100000|   200|  150|   Alabama|   NULL|        13|
| 18|     Laila|     Mark| 26|  F|         Sales|     Sales|100000|   200|  150|   Florida|   23St|        11|
|

In [ ]:
dept_df = df.select('department','salary').orderBy(col("salary").desc())

In [ ]:
window = Window.partitionBy('department').orderBy(col('salary').desc())

In [ ]:
from pyspark.sql.functions.builtin import rank
final_df = dept_df.withColumn('rank', rank().over(window))

In [ ]:
final_df.show()

+----------+------+----+
|department|salary|rank|
+----------+------+----+
|     Audit|110000|   1|
|     Audit|100000|   2|
|     Audit|100000|   2|
|     Audit| 70000|   4|
|Management|250000|   1|
|Management|200000|   2|
|Management|150000|   3|
|Management|100000|   4|
|     Sales|220000|   1|
|     Sales|200000|   2|
|     Sales|140000|   3|
|     Sales|130000|   4|
|     Sales|130000|   4|
|     Sales|100000|   6|
|     Sales|100000|   6|
|     Sales|  NULL|   8|
+----------+------+----+



In [ ]:
# Filter the top 3 salaries for each department
top_salaries_df = final_df.filter(col("rank") <= 3)

# Sort results by department name and salary in descending order
result_df = top_salaries_df.orderBy("department", col("salary").desc())

# Show the final results
result_df.show(truncate=False)

+----------+------+----+
|department|salary|rank|
+----------+------+----+
|Audit     |110000|1   |
|Audit     |100000|2   |
|Audit     |100000|2   |
|Management|250000|1   |
|Management|200000|2   |
|Management|150000|3   |
|Sales     |220000|1   |
|Sales     |200000|2   |
|Sales     |140000|3   |
+----------+------+----+



# Question 4
Given two datasets: one containing signup details (including start and stop times) and another containing transaction details (such as amounts), determine the most profitable location based on signup duration and transaction amounts.

In [ ]:
signups_data = [
    (1, '2020-01-01 10:00:00', '2020-01-01 12:00:00', 101, 'New York'),
    (2, '2020-01-02 11:00:00', '2020-01-02 13:00:00', 102, 'Los Angeles'),
    (3, '2020-01-03 10:00:00', '2020-01-03 14:00:00', 103, 'Chicago'),
    (4, '2020-01-04 09:00:00', '2020-01-04 10:30:00', 101, 'San Francisco'),
    (5, '2020-01-05 08:00:00', '2020-01-05 11:00:00', 102, 'New York')
]
transactions_data = [
    (1, 1, '2020-01-01 10:30:00', 50.00),
    (2, 1, '2020-01-01 11:00:00', 30.00),
    (3, 2, '2020-01-02 11:30:00', 100.00),
    (4, 2, '2020-01-02 12:00:00', 75.00),
    (5, 3, '2020-01-03 10:30:00', 120.00),
    (6, 4, '2020-01-04 09:15:00', 80.00),
    (7, 5, '2020-01-05 08:30:00', 90.00)
]
# Define columns for signups DataFrame
signups_columns = [
  "signup_id",
  "signup_start_date",
  "signup_stop_date",
  "plan_id",
  "location"
]
signups_df = spark.createDataFrame(signups_data, signups_columns)

# Define columns for transactions DataFrame
transactions_columns = [
  "transaction_id",
  "signup_id",
  "transaction_start_date",
  "amt"
]
transactions_df = spark.createDataFrame(transactions_data, transactions_columns)


In [ ]:
signups_df = signups_df.withColumn(
    "signup_duration_minutes",
    (unix_timestamp("signup_stop_date") - unix_timestamp("signup_start_date")) / 60
)

In [ ]:
transactions_df.show()

+--------------+---------+----------------------+-----+
|transaction_id|signup_id|transaction_start_date|  amt|
+--------------+---------+----------------------+-----+
|             1|        1|   2020-01-01 10:30:00| 50.0|
|             2|        1|   2020-01-01 11:00:00| 30.0|
|             3|        2|   2020-01-02 11:30:00|100.0|
|             4|        2|   2020-01-02 12:00:00| 75.0|
|             5|        3|   2020-01-03 10:30:00|120.0|
|             6|        4|   2020-01-04 09:15:00| 80.0|
|             7|        5|   2020-01-05 08:30:00| 90.0|
+--------------+---------+----------------------+-----+



In [ ]:
signups_df.show()

+---------+-------------------+-------------------+-------+-------------+-----------------------+
|signup_id|  signup_start_date|   signup_stop_date|plan_id|     location|signup_duration_minutes|
+---------+-------------------+-------------------+-------+-------------+-----------------------+
|        1|2020-01-01 10:00:00|2020-01-01 12:00:00|    101|     New York|                  120.0|
|        2|2020-01-02 11:00:00|2020-01-02 13:00:00|    102|  Los Angeles|                  120.0|
|        3|2020-01-03 10:00:00|2020-01-03 14:00:00|    103|      Chicago|                  240.0|
|        4|2020-01-04 09:00:00|2020-01-04 10:30:00|    101|San Francisco|                   90.0|
|        5|2020-01-05 08:00:00|2020-01-05 11:00:00|    102|     New York|                  180.0|
+---------+-------------------+-------------------+-------+-------------+-----------------------+



In [ ]:
combined_df = signups_df.join(transactions_df, on='signup_id', how='inner')

In [ ]:
combined_df.show()

+---------+-------------------+-------------------+-------+-------------+-----------------------+--------------+----------------------+-----+
|signup_id|  signup_start_date|   signup_stop_date|plan_id|     location|signup_duration_minutes|transaction_id|transaction_start_date|  amt|
+---------+-------------------+-------------------+-------+-------------+-----------------------+--------------+----------------------+-----+
|        1|2020-01-01 10:00:00|2020-01-01 12:00:00|    101|     New York|                  120.0|             1|   2020-01-01 10:30:00| 50.0|
|        1|2020-01-01 10:00:00|2020-01-01 12:00:00|    101|     New York|                  120.0|             2|   2020-01-01 11:00:00| 30.0|
|        2|2020-01-02 11:00:00|2020-01-02 13:00:00|    102|  Los Angeles|                  120.0|             3|   2020-01-02 11:30:00|100.0|
|        2|2020-01-02 11:00:00|2020-01-02 13:00:00|    102|  Los Angeles|                  120.0|             4|   2020-01-02 12:00:00| 75.0|
|     

In [ ]:
# Calculate average transaction amount for each signup
transaction_avg_df = transactions_df.groupBy("signup_id").agg(avg("amt").alias("avg_transaction_amount"))

# Join the signups with transaction averages
joined_df = signups_df.join(transaction_avg_df, on="signup_id", how="inner")

# Group by location and calculate average duration, average transaction amount, and ratio
result_df = joined_df.groupBy("location").agg(
    avg("signup_duration_minutes").alias("avg_duration"),
    avg("avg_transaction_amount").alias("avg_transaction_amount")
)

In [ ]:
result_df.show()

+-------------+------------+----------------------+
|     location|avg_duration|avg_transaction_amount|
+-------------+------------+----------------------+
|  Los Angeles|       120.0|                  87.5|
|San Francisco|        90.0|                  80.0|
|      Chicago|       240.0|                 120.0|
|     New York|       150.0|                  65.0|
+-------------+------------+----------------------+



In [ ]:
result_df = result_df.withColumn(
    "ratio",
    when(col("avg_duration") != 0, col("avg_transaction_amount") / col("avg_duration")).otherwise(0)
)

# Sort by ratio from highest to lowest
result_df = result_df.orderBy(col("ratio"), ascending=False)

# Show the final result
result_df.show(truncate=False)

+-------------+------------+----------------------+-------------------+
|location     |avg_duration|avg_transaction_amount|ratio              |
+-------------+------------+----------------------+-------------------+
|San Francisco|90.0        |80.0                  |0.8888888888888888 |
|Los Angeles  |120.0       |87.5                  |0.7291666666666666 |
|Chicago      |240.0       |120.0                 |0.5                |
|New York     |150.0       |65.0                  |0.43333333333333335|
+-------------+------------+----------------------+-------------------+



The problem is to calculate the minimum number of platforms required at a train station based on the given arrival_times and departure_times.

In [ ]:
arrivals_data = [
    (1, '2024-11-17 08:00'),
    (2, '2024-11-17 08:05'),
    (3, '2024-11-17 08:05'),
    (4, '2024-11-17 08:10'),
    (5, '2024-11-17 08:10'),
    (6, '2024-11-17 12:15'),
    (7, '2024-11-17 12:20'),
    (8, '2024-11-17 12:25'),
    (9, '2024-11-17 15:00'),
    (10, '2024-11-17 15:00'),
    (11, '2024-11-17 15:00'),
    (12, '2024-11-17 15:06'),
    (13, '2024-11-17 20:00'),
    (14, '2024-11-17 20:10')
]
departures_data = [
    (1, '2024-11-17 08:15'),
    (2, '2024-11-17 08:10'),
    (3, '2024-11-17 08:20'),
    (4, '2024-11-17 08:25'),
    (5, '2024-11-17 08:20'),
    (6, '2024-11-17 13:00'),
    (7, '2024-11-17 12:25'),
    (8, '2024-11-17 12:30'),
    (9, '2024-11-17 15:05'),
    (10, '2024-11-17 15:10'),
    (11, '2024-11-17 15:15'),
    (12, '2024-11-17 15:15'),
    (13, '2024-11-17 20:15'),
    (14, '2024-11-17 20:15')
]
# Define schema for the data
arrival_columns = ['train_id', 'arrival_time']
departure_columns = ['train_id', 'departure_time']
# Create DataFrames
arrivals_df = spark.createDataFrame(arrivals_data, arrival_columns)
departures_df = spark.createDataFrame(departures_data, departure_columns)

# Convert the time strings to timestamps for easier handling
arrivals_df = arrivals_df.withColumn('arrival_time', F.col('arrival_time').cast('timestamp'))
departures_df = departures_df.withColumn('departure_time', F.col('departure_time').cast('timestamp'))

# Add event type (arrival = 1, departure = -1)
arrivals_df = arrivals_df.withColumn('event_type', F.lit(1))
departures_df = departures_df.withColumn('event_type', F.lit(-1))

# Union both DataFrames into one, marking events as either arrival or departure
all_events_df = arrivals_df.select('train_id', 'arrival_time', 'event_type') \
    .withColumnRenamed('arrival_time', 'event_time') \
    .union(departures_df.select('train_id', 'departure_time', 'event_type') \
    .withColumnRenamed('departure_time', 'event_time'))



In [ ]:
all_events_df.show()

+--------+-------------------+----------+
|train_id|         event_time|event_type|
+--------+-------------------+----------+
|       1|2024-11-17 08:00:00|         1|
|       2|2024-11-17 08:05:00|         1|
|       3|2024-11-17 08:05:00|         1|
|       4|2024-11-17 08:10:00|         1|
|       5|2024-11-17 08:10:00|         1|
|       6|2024-11-17 12:15:00|         1|
|       7|2024-11-17 12:20:00|         1|
|       8|2024-11-17 12:25:00|         1|
|       9|2024-11-17 15:00:00|         1|
|      10|2024-11-17 15:00:00|         1|
|      11|2024-11-17 15:00:00|         1|
|      12|2024-11-17 15:06:00|         1|
|      13|2024-11-17 20:00:00|         1|
|      14|2024-11-17 20:10:00|         1|
|       1|2024-11-17 08:15:00|        -1|
|       2|2024-11-17 08:10:00|        -1|
|       3|2024-11-17 08:20:00|        -1|
|       4|2024-11-17 08:25:00|        -1|
|       5|2024-11-17 08:20:00|        -1|
|       6|2024-11-17 13:00:00|        -1|
+--------+-------------------+----

In [ ]:
all_events_df = all_events_df.orderBy('event_time',F.col('event_type').desc())

In [ ]:
window_spec = Window.orderBy('event_time', F.col('event_type').desc())

In [ ]:
all_events_df = all_events_df.withColumn('platform_needed', F.sum('event_type').over(window_spec))

In [ ]:
max_platforms = all_events_df.agg(F.max('platform_needed')).collect()

# Question 8
We have two datasets:

Sessions Table: Contains records of when users started their sessions.
Order Summary Table: Contains records of orders placed by users along with their values.
We want to:

Find users who started a session and placed an order on the same day.
Calculate the total number of orders and the total order value for those users.

In [ ]:
sessions_data = [
    (1, 1, '2024-01-01 00:00:00'),
    (2, 2, '2024-01-02 00:00:00'),
    (3, 3, '2024-01-05 00:00:00'),
    (4, 3, '2024-01-05 00:00:00'),
    (5, 4, '2024-01-03 00:00:00'),
    (6, 4, '2024-01-03 00:00:00'),
    (7, 5, '2024-01-04 00:00:00'),
    (8, 5, '2024-01-04 00:00:00'),
    (9, 3, '2024-01-05 00:00:00'),
    (10, 5, '2024-01-04 00:00:00')
]
# Sample data for orders
orders_data = [
    (1, 1, 152, '2024-01-01 00:00:00'),
    (2, 2, 485, '2024-01-02 00:00:00'),
    (3, 3, 398, '2024-01-05 00:00:00'),
    (4, 3, 320, '2024-01-05 00:00:00'),
    (5, 4, 156, '2024-01-03 00:00:00'),
    (6, 4, 121, '2024-01-03 00:00:00'),
    (7, 5, 238, '2024-01-04 00:00:00'),
    (8, 5, 70, '2024-01-04 00:00:00'),
    (9, 3, 152, '2024-01-05 00:00:00'),
    (10, 5, 171, '2024-01-04 00:00:00')
]
session_columns = ["session_id", "user_id", "session_date"]
orders_columns = ["order_id", "user_id", "order_value", "order_date"]
# Convert data into DataFrames
sessions_df = spark.createDataFrame(sessions_data, session_columns)
orders_df = spark.createDataFrame(orders_data, orders_columns)

In [ ]:
combined_df = sessions_df.join(orders_df, on='user_id', how='inner')

In [ ]:
combined_df.show()

+-------+----------+-------------------+--------+-----------+-------------------+
|user_id|session_id|       session_date|order_id|order_value|         order_date|
+-------+----------+-------------------+--------+-----------+-------------------+
|      1|         1|2024-01-01 00:00:00|       1|        152|2024-01-01 00:00:00|
|      2|         2|2024-01-02 00:00:00|       2|        485|2024-01-02 00:00:00|
|      3|         3|2024-01-05 00:00:00|       3|        398|2024-01-05 00:00:00|
|      3|         3|2024-01-05 00:00:00|       4|        320|2024-01-05 00:00:00|
|      3|         3|2024-01-05 00:00:00|       9|        152|2024-01-05 00:00:00|
|      3|         4|2024-01-05 00:00:00|       3|        398|2024-01-05 00:00:00|
|      3|         4|2024-01-05 00:00:00|       4|        320|2024-01-05 00:00:00|
|      3|         4|2024-01-05 00:00:00|       9|        152|2024-01-05 00:00:00|
|      3|         9|2024-01-05 00:00:00|       3|        398|2024-01-05 00:00:00|
|      3|       

In [ ]:
combined_df = combined_df.withColumn('session_date', F.col('session_date').cast('date'))
combined_df = combined_df.withColumn('order_date', F.col('order_date').cast('date'))

In [ ]:
from  pyspark.sql.functions import datediff
combined_df = combined_df.withColumn('Days_between', datediff("session_date", "order_date"))

In [ ]:
combined_df.select('user_id').filter(col('Days_between')==0).distinct().show()

+-------+
|user_id|
+-------+
|      1|
|      2|
|      3|
|      4|
|      5|
+-------+



# Question

We have to find the 3rd highest total transaction amount from the records. We have two tables: one containing customer details (customers) and the other storing transaction data (card_orders).

Our goal is to retrieve the customer who ranks third in terms of total transaction amount.

In [ ]:
customers_data = [
    (1, 'Jill', 'Doe', 'New York', '123 Main St', '555-1234'),
    (2, 'Henry', 'Smith', 'Los Angeles', '456 Oak Ave', '555-5678'),
    (3, 'William', 'Johnson', 'Chicago', '789 Pine Rd', '555-8765'),
    (4, 'Emma', 'Daniel', 'Houston', '321 Maple Dr', '555-4321'),
    (5, 'Charlie', 'Davis', 'Phoenix', '654 Elm St', '555-6789')
]
customers_columns = [
  'id',
  'first_name',
  'last_name',
  'city',
  'address',
  'phone_number'
]
customers_df = spark.createDataFrame(customers_data, customers_columns)
# Create the card_orders DataFrame
card_orders_data = [
    (1, 1, '2024-11-01 10:00:00', 'Electronics', 200),
    (2, 2, '2024-11-02 11:30:00', 'Groceries', 150),
    (3, 1, '2024-11-03 15:45:00', 'Clothing', 120),
    (4, 3, '2024-11-04 09:10:00', 'Books', 90),
    (8, 3, '2024-11-08 10:20:00', 'Groceries', 130),
    (9, 1, '2024-11-09 12:00:00', 'Books', 180),
    (10, 4, '2024-11-10 11:15:00', 'Electronics', 200),
    (11, 5, '2024-11-11 14:45:00', 'Furniture', 150),
    (12, 2, '2024-11-12 09:30:00', 'Furniture', 180)
]
card_orders_columns = [
  'order_id',
  'cust_id',
  'order_date',
  'order_details',
  'total_order_cost'
]
card_orders_df = spark.createDataFrame(card_orders_data, card_orders_columns)


In [ ]:
combined_df = card_orders_df.join(customers_df, customers_df.id == card_orders_df.cust_id)

In [ ]:
grouped_df = combined_df.groupBy('cust_id').agg(F.sum('total_order_cost'))

In [ ]:
windows_spec = Window.partitionBy('cust_id').orderBy(F.col('sum(total_order_cost)').desc())

In [ ]:
window_spec = Window.orderBy(F.desc('total_transaction_amount'))
ranked_transactions = grouped_df.withColumn('rank', F.rank().over(window_spec))

# Select the customer with the third-highest total transaction amount
third_highest_customer = ranked_transactions.filter(ranked_transactions.rank == 3) \
    .select('id', 'first_name', 'last_name')
third_highest_customer.show()

{"ts": "2026-04-16 06:13:21.136", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `total_transaction_amount` cannot be resolved. Did you mean one of the following? [`sum(total_order_cost)`, `cust_id`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "desc", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o2539.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `total_transaction_amount` cannot be resolved. Did you mean one of the following? [`sum(total_order_cost)`, `cust_id`]. SQLSTATE: 42703;\n'Project [cust_id#829L, sum(total_order_cost)#971L, rank() windowspecdefinition('total_transaction_amount DESC NULLS LAS

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `total_transaction_amount` cannot be resolved. Did you mean one of the following? [`sum(total_order_cost)`, `cust_id`]. SQLSTATE: 42703;
'Project [cust_id#829L, sum(total_order_cost)#971L, rank() windowspecdefinition('total_transaction_amount DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rank#993]
+- Aggregate [cust_id#829L], [cust_id#829L, sum(total_order_cost#832L) AS sum(total_order_cost)#971L]
   +- Join Inner, (id#822L = cust_id#829L)
      :- LogicalRDD [order_id#828L, cust_id#829L, order_date#830, order_details#831, total_order_cost#832L], false
      +- LogicalRDD [id#822L, first_name#823, last_name#824, city#825, address#826, phone_number#827], false


### Question 1
Given a clickstream of user activity data, find the relevant user session for each click event.
'''
+----------------------+---------+
| click_time           | user_id |
+----------------------+---------+
| 2018–01–01 11:00:00  | u1      |
| 2018–01–01 12:00:00  | u1      |
| 2018–01–01 13:00:00  | u1      |
| 2018–01–01 13:00:00  | u1      |
| 2018–01–01 14:00:00  | u1      |
| 2018–01–01 15:00:00  | u1      |
| 2018–01–01 11:00:00  | u2      |
| 2018–01–02 11:00:00  | u2      |
+----------------------+---------+'''
Session definition:

session expires after inactivity of 30mins, because of inactivity no clickstream will be generated
Session remains active for a total of 2 hours

In [ ]:
schema = "click_time STRING, user_id STRING"

data = [
    ("2018-01-01 11:00:00", "u1"),
    ("2018-01-01 12:00:00", "u1"),
    ("2018-01-01 13:00:00", "u1"),
    ("2018-01-01 13:00:00", "u1"),
    ("2018-01-01 14:00:00", "u1"),
    ("2018-01-01 15:00:00", "u1"),
    ("2018-01-01 11:00:00", "u2"),
    ("2018-01-02 11:00:00", "u2")
]

clickstream_df = spark.createDataFrame(data, schema=schema)

In [ ]:
clickstream_df.show()

+-------------------+-------+---------------+--------------------+
|         click_time|user_id|click_timestamp|prev_click_timestamp|
+-------------------+-------+---------------+--------------------+
|2018-01-01 11:00:00|     u1|     1514804400|                NULL|
|2018-01-01 12:00:00|     u1|     1514808000|          1514804400|
|2018-01-01 13:00:00|     u1|     1514811600|          1514808000|
|2018-01-01 13:00:00|     u1|     1514811600|          1514811600|
|2018-01-01 14:00:00|     u1|     1514815200|          1514811600|
|2018-01-01 15:00:00|     u1|     1514818800|          1514815200|
|2018-01-01 11:00:00|     u2|     1514804400|                NULL|
|2018-01-02 11:00:00|     u2|     1514890800|          1514804400|
+-------------------+-------+---------------+--------------------+



In [ ]:
# Convert click_time to Unix timestamp for easier calculations
clickstream_df = clickstream_df.withColumn("click_timestamp", unix_timestamp("click_time"))
session_window = Window.partitionBy("user_id").orderBy("click_timestamp")


In [ ]:
clickstream_df = clickstream_df.withColumn("prev_click_timestamp", F.lag("click_timestamp", 1).over(session_window))

In [ ]:
clickstream_df = clickstream_df.withColumn("timestamp_diff", (col("click_timestamp")-col("prev_click_timestamp"))/60)
# Updating null with 0
clickstream_df = clickstream_df.withColumn("timestamp_diff", when(col("timestamp_diff").isNull(), 0).otherwise(col("timestamp_diff")))
# Check for new session
clickstream_df = clickstream_df.withColumn("session_new", F.when(col("timestamp_diff") > 30, 1).otherwise(0))
# New session names
clickstream_df = clickstream_df.withColumn("session_new_name", F.concat(col("user_id"), F.lit("--S"), sum(col("session_new")).over(session_window)))
clickstream_df.show()

PySparkTypeError: [NOT_ITERABLE] Column is not iterable.